# 02 - Data Cleaning

## Objective

This notebook cleans the original hotel booking dataset before the preprocessing and model training stage.

The goal of this step is to improve data quality while keeping the cleaning decisions reasonable and explainable. Each cleaning action is based on the characteristics of the dataset and the objective of predicting booking cancellations.

Main cleaning tasks:

- Load the original hotel booking dataset.
- Inspect data types, missing values, and duplicated rows.
- Check exact duplicated rows without removing them, because the dataset does not contain a unique booking ID or customer ID.
- Handle missing values in `children`, `country`, `agent`, and `company`.
- Convert basic data types after handling missing values.
- Standardize text columns by removing unnecessary spaces.
- Handle unclear categorical values such as `"Undefined"`.
- Create and convert date-related variables.
- Create booking-related variables such as `total_guests` and `total_nights`.
- Remove logically invalid records, such as bookings with zero guests or zero nights.
- Handle abnormal ADR values by removing negative ADR and extremely high ADR outliers.
- Check lead time and guest-related values without removing them automatically.
- Drop potential data leakage columns before modeling.
- Save the cleaned modeling dataset for the next notebook.

## 1. Import Libraries

In this step, we import the required Python libraries for data cleaning.

`pandas` is used for loading, inspecting, and manipulating the dataset.  
`numpy` is used for numerical operations and for creating new variables during the cleaning process.

In [1]:
import pandas as pd
import numpy as np

## 2. Load the Original Dataset

Although the dataset was already loaded in the previous notebook, this notebook is designed to run independently. Therefore, the original dataset is loaded again.

A copy named `df_clean` is created so that the original dataset remains unchanged during the cleaning process.

In [2]:
df = pd.read_csv("../Data/hotel_bookings.csv")

df_clean = df.copy()

print("Original dataset shape:", df_clean.shape)
df_clean.head()

Original dataset shape: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 3. Initial Data Inspection

Before cleaning the dataset, we first inspect its general structure.

This step includes checking the dataset shape, data types, missing values, and exact duplicated rows. The goal is to identify the main data quality issues that need to be handled in the cleaning process.

In [3]:
print("Dataset shape:", df_clean.shape)

Dataset shape: (119390, 32)


In [4]:
print("\nData types:")
print(df_clean.dtypes)


Data types:
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                     int64
deposit_type                          str
agent                

In [5]:
print("\nMissing values:")
print(df_clean.isnull().sum().sort_values(ascending=False))


Missing values:
company                           112593
agent                              16340
country                              488
children                               4
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
hotel                                  0
previous_cancellations                 0
days_in_waiting_list                   0
customer_type                          0
adr                                    0
required_car_parking_spaces            0
total_of_special_requests              0
reservation_status                     0
previous_bookings_not_canceled         0
is_repeated_guest                      0
is_canceled                            0
distribution_channel                   0
market_segment                         0
meal                                   0
babies                                 0
adults                                 0

## 4. Check Duplicated Rows

The dataset contains exact duplicated rows. However, the dataset does not include a unique booking ID or customer ID.

Because of this, we cannot confirm whether these identical rows are real duplicated records or separate bookings with the same characteristics.

Therefore, duplicated rows are not removed in this project. They are only reported for awareness.

In [6]:
duplicate_count = df_clean.duplicated().sum()
duplicate_percent = duplicate_count / len(df_clean) * 100

print("Total rows:", len(df_clean))
print("Exact duplicated rows:", duplicate_count)
print(f"Duplicate percentage: {duplicate_percent:.2f}%")

Total rows: 119390
Exact duplicated rows: 31994
Duplicate percentage: 26.80%


## 5. Handle Missing Values and Convert Basic Data Types

The dataset contains missing values in several columns: `children`, `country`, `agent`, and `company`.

The following cleaning decisions were applied:

- Missing values in `children` are replaced with 0, assuming that missing values mean no children.
- Missing values in `country` are replaced with `"Unknown"` to avoid losing observations.
- Missing values in `agent` are replaced with 0, meaning the booking was not made through an agent.
- Missing values in `company` are replaced with 0, meaning the booking was not associated with a company.

Two additional binary variables were also created to preserve information about whether an agent or company was involved:

- `has_agent`: indicates whether the booking used an agent.
- `has_company`: indicates whether the booking was associated with a company.

After handling missing values, `children`, `agent`, and `company` are converted into integer type because they represent counts or ID-like values.

Date-related type conversion is handled later in the date feature engineering step.

In [7]:
df_clean["children"] = df_clean["children"].fillna(0)

df_clean["country"] = df_clean["country"].fillna("Unknown")

df_clean["agent"] = df_clean["agent"].fillna(0)
df_clean["company"] = df_clean["company"].fillna(0)

df_clean["has_agent"] = np.where(df_clean["agent"] == 0, 0, 1)
df_clean["has_company"] = np.where(df_clean["company"] == 0, 0, 1)

df_clean["children"] = df_clean["children"].astype(int)
df_clean["agent"] = df_clean["agent"].astype(int)
df_clean["company"] = df_clean["company"].astype(int)

print(df_clean[["children", "country", "agent", "company", "has_agent", "has_company"]].isnull().sum())

children       0
country        0
agent          0
company        0
has_agent      0
has_company    0
dtype: int64


## 6. Standardize Text Columns

Some categorical columns may contain unnecessary spaces before or after the text values.

To make the categories consistent, leading and trailing spaces are removed from all text columns.

This step is performed after handling missing values to avoid converting missing values into string values such as `"nan"`.

In [8]:
text_cols = [
    col for col in df_clean.columns
    if df_clean[col].map(type).eq(str).any()
]

for col in text_cols:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

print("Text columns standardized:", text_cols)

Text columns standardized: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type', 'reservation_status', 'reservation_status_date']


## 7. Handle Undefined Categorical Values

Some categorical columns contain the value `"Undefined"`. Although this is not a missing value in the technical sense, it still represents unclear or unspecified information.

In the `meal` column, `"Undefined"` is replaced with `"SC"`. `"SC"` means self-catering, which indicates that no meal package is included. Therefore, undefined meal values are treated as bookings with no clearly specified meal package.

For `market_segment` and `distribution_channel`, `"Undefined"` is replaced with `"Unknown"`. These columns describe how the booking was made, and it would not be appropriate to assign undefined values to a specific booking channel without evidence.

This step helps make categorical variables more consistent and easier to interpret during analysis and modeling.

In [9]:
df_clean["meal"] = df_clean["meal"].replace("Undefined", "SC")

df_clean["market_segment"] = df_clean["market_segment"].replace("Undefined", "Unknown")
df_clean["distribution_channel"] = df_clean["distribution_channel"].replace("Undefined", "Unknown")

print("Meal values after cleaning:")
print(df_clean["meal"].value_counts())

print("\nMarket segment values after cleaning:")
print(df_clean["market_segment"].value_counts())

print("\nDistribution channel values after cleaning:")
print(df_clean["distribution_channel"].value_counts())

Meal values after cleaning:
meal
BB    92310
HB    14463
SC    11819
FB      798
Name: count, dtype: int64

Market segment values after cleaning:
market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Unknown              2
Name: count, dtype: int64

Distribution channel values after cleaning:
distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Unknown          5
Name: count, dtype: int64


## 8. Create and Convert Date Variables

The original dataset stores the arrival date in three separate columns: `arrival_date_year`, `arrival_date_month`, and `arrival_date_day_of_month`.

To make date-based analysis easier, these columns are combined into a new datetime variable named `arrival_date`.

The `reservation_status_date` column is also converted into datetime format in this step. Date-related type conversion is handled here because it is part of date feature engineering.

The parameter `errors="coerce"` is used so that invalid dates, if any, will be converted into missing values instead of causing an error. These invalid dates will be checked and handled in the next step.

In [10]:
# Create a mapping from month names to month numbers
month_map = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12
}

# Convert arrival month names into month numbers
df_clean["arrival_month_num"] = df_clean["arrival_date_month"].map(month_map)

# Create a complete arrival date column
df_clean["arrival_date"] = pd.to_datetime(
    dict(
        year=df_clean["arrival_date_year"],
        month=df_clean["arrival_month_num"],
        day=df_clean["arrival_date_day_of_month"]
    ),
    errors="coerce"
)

# Convert reservation_status_date into datetime format
df_clean["reservation_status_date"] = pd.to_datetime(
    df_clean["reservation_status_date"],
    errors="coerce"
)

# Check whether any invalid dates were created
print("Missing values in date columns after conversion:")
print(df_clean[["arrival_date", "reservation_status_date"]].isnull().sum())

# Preview date columns
df_clean[[
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_day_of_month",
    "arrival_date",
    "reservation_status_date"
]].head()

Missing values in date columns after conversion:
arrival_date               0
reservation_status_date    0
dtype: int64


,arrival_date_year,arrival_date_month,arrival_date_day_of_month,arrival_date,reservation_status_date
0,2015,July,1,2015-07-01,2015-07-01
1,2015,July,1,2015-07-01,2015-07-01
2,2015,July,1,2015-07-01,2015-07-02
3,2015,July,1,2015-07-01,2015-07-02
4,2015,July,1,2015-07-01,2015-07-03


In [11]:
# Check data types after date conversion
print("\nData types of date columns after conversion:")
print(df_clean[["arrival_date", "reservation_status_date"]].dtypes)


Data types of date columns after conversion:
arrival_date               datetime64[us]
reservation_status_date    datetime64[us]
dtype: object


## 9. Create Booking-Related Variables

In this step, two new variables are created to better understand each booking:

- `total_guests`: the total number of guests in a booking.
- `total_nights`: the total number of nights stayed.

`total_guests` is calculated by adding the number of adults, children, and babies.

`total_nights` is calculated by adding weekend nights and week nights.

These variables are useful for checking whether a booking is logically valid. For example, a booking with zero guests or zero nights may not represent a valid hotel stay.

In [12]:
# Create total number of guests
df_clean["total_guests"] = (
    df_clean["adults"] +
    df_clean["children"] +
    df_clean["babies"]
)

# Create total number of nights
df_clean["total_nights"] = (
    df_clean["stays_in_weekend_nights"] +
    df_clean["stays_in_week_nights"]
)

# Check logically unusual bookings
print("Bookings with 0 guest:", (df_clean["total_guests"] == 0).sum())
print("Bookings with 0 night:", (df_clean["total_nights"] == 0).sum())

# Preview new variables
df_clean[
    [
        "adults",
        "children",
        "babies",
        "total_guests",
        "stays_in_weekend_nights",
        "stays_in_week_nights",
        "total_nights"
    ]
].head()

Bookings with 0 guest: 180
Bookings with 0 night: 715


,adults,children,babies,total_guests,stays_in_weekend_nights,stays_in_week_nights,total_nights
0,2,0,0,2,0,0,0
1,2,0,0,2,0,0,0
2,1,0,0,1,0,1,1
3,1,0,0,1,0,1,1
4,2,0,0,2,0,2,2


## 10. Remove Logically Invalid Records

After creating `total_guests` and `total_nights`, some logically invalid records were identified.

The following records are removed:

- Bookings with `total_guests = 0`, because a booking with no guests does not represent a valid hotel stay.
- Bookings with `total_nights = 0`, because a booking with no stay duration is not suitable for hotel stay analysis.

Removing these records helps improve the quality and consistency of the dataset before outlier handling and modeling.

In [13]:
before_rows = df_clean.shape[0]

df_clean = df_clean[df_clean["total_guests"] > 0]
df_clean = df_clean[df_clean["total_nights"] > 0]

df_clean = df_clean.reset_index(drop=True)

after_rows = df_clean.shape[0]

print("Rows before removing invalid records:", before_rows)
print("Rows after removing invalid records:", after_rows)
print("Rows removed:", before_rows - after_rows)

print("\nRemaining bookings with 0 guest:", (df_clean["total_guests"] == 0).sum())
print("Remaining bookings with 0 night:", (df_clean["total_nights"] == 0).sum())

Rows before removing invalid records: 119390
Rows after removing invalid records: 118565
Rows removed: 825

Remaining bookings with 0 guest: 0
Remaining bookings with 0 night: 0


## 11. Handle Abnormal Values in ADR

`adr` stands for Average Daily Rate, which represents the average price per night.

Negative ADR values are not logically valid because room prices cannot be negative. Therefore, records with negative ADR values are removed.

Extremely high ADR values may represent abnormal or exceptional cases that can distort the analysis and affect model performance. Therefore, ADR values above the 99th percentile are treated as outliers and removed from the dataset.

This step helps reduce the influence of extreme price values and makes the dataset more suitable for analysis and modeling.

In [14]:
# Check ADR before cleaning
print("ADR summary before cleaning:")
print(df_clean["adr"].describe())

# Check negative ADR values
print("\nBookings with negative ADR:")
print(df_clean[df_clean["adr"] < 0][["hotel", "adr", "is_canceled"]])

# Remove negative ADR values
df_clean = df_clean[df_clean["adr"] >= 0].reset_index(drop=True)

# Define ADR upper threshold using the 99th percentile
adr_upper = df_clean["adr"].quantile(0.99)

print("\nADR upper threshold:", adr_upper)

# Remove extremely high ADR values
before_adr_outliers = df_clean.shape[0]

df_clean = df_clean[df_clean["adr"] <= adr_upper].reset_index(drop=True)

after_adr_outliers = df_clean.shape[0]

print("Rows before removing high ADR outliers:", before_adr_outliers)
print("Rows after removing high ADR outliers:", after_adr_outliers)
print("Rows removed:", before_adr_outliers - after_adr_outliers)

# Check ADR after cleaning
print("\nADR summary after cleaning:")
print(df_clean["adr"].describe())

print("\nBookings with negative ADR after cleaning:", (df_clean["adr"] < 0).sum())
print("Bookings above ADR upper threshold after cleaning:", (df_clean["adr"] > adr_upper).sum())

ADR summary before cleaning:
count    118565.000000
mean        102.523809
std          50.005542
min          -6.380000
25%          70.000000
50%          95.000000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

Bookings with negative ADR:
              hotel   adr  is_canceled
14879  Resort Hotel -6.38            0

ADR upper threshold: 252.0
Rows before removing high ADR outliers: 118564
Rows after removing high ADR outliers: 117396
Rows removed: 1168

ADR summary after cleaning:
count    117396.000000
mean        100.661356
std          44.050703
min           0.000000
25%          70.000000
50%          94.500000
75%         125.000000
max         252.000000
Name: adr, dtype: float64

Bookings with negative ADR after cleaning: 0
Bookings above ADR upper threshold after cleaning: 0


## 12. Check Lead Time Values

`lead_time` represents the number of days between the booking date and the arrival date.

Very high lead time values may look unusual, but they are not necessarily data errors because some customers may book far in advance.

Therefore, lead time values are checked but not removed in this step.

In [16]:
print("Lead time summary:")
print(df_clean["lead_time"].describe())

lead_time_99 = df_clean["lead_time"].quantile(0.99)

print("\nLead time 99th percentile:", lead_time_99)
print("Bookings above 99th percentile:", (df_clean["lead_time"] > lead_time_99).sum())

Lead time summary:
count    117396.000000
mean        104.852448
std         107.161278
min           0.000000
25%          18.000000
50%          70.000000
75%         162.000000
max         709.000000
Name: lead_time, dtype: float64

Lead time 99th percentile: 445.1000000000058
Bookings above 99th percentile: 1174


## 13. Check Unusual Guest-Related Values

Some bookings may have unusually high numbers of adults, children, or babies.

These values are not removed automatically because they may represent valid group bookings or family bookings.

Therefore, this step only checks and reports the distribution of guest-related variables for review.

In [17]:
# Check guest-related variables

guest_cols = ["adults", "children", "babies", "total_guests"]

for col in guest_cols:
    print(f"\n{col}:")
    print(df_clean[col].describe())
    print("Top values:")
    print(df_clean[col].value_counts().sort_index().tail(10))


adults:
count    117396.000000
mean          1.856392
std           0.574972
min           0.000000
25%           2.000000
50%           2.000000
75%           2.000000
max          55.000000
Name: adults, dtype: float64
Top values:
adults
4     47
5      2
6      1
10     1
20     2
26     5
27     2
40     1
50     1
55     1
Name: count, dtype: int64

children:
count    117396.000000
mean          0.095327
std           0.379847
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          10.000000
Name: children, dtype: float64
Top values:
children
0     109526
1       4607
2       3212
3         50
10         1
Name: count, dtype: int64

babies:
count    117396.000000
mean          0.007803
std           0.096837
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          10.000000
Name: babies, dtype: float64
Top values:
babies
0     116512
1        867
2         15
9          1
10         1

## 14. Drop Potential Data Leakage Columns

The modeling objective is to predict whether a booking will be canceled using the target variable `is_canceled`.

Two columns may cause data leakage:

- `reservation_status`
- `reservation_status_date`

These columns contain information that would not be available at the time of prediction.

The `reservation_status` column directly reveals the final booking outcome, such as canceled, checked out, or no-show. If this column is kept, the model may learn the answer directly instead of making a real prediction.

The `reservation_status_date` column may also contain future information because it records the date when the final booking status was updated.

Therefore, these columns are removed from the modeling dataset to avoid unrealistic model performance.

In [18]:
# Drop potential data leakage columns for modeling

leakage_cols = ["reservation_status", "reservation_status_date"]

df_clean_model = df_clean.drop(columns=leakage_cols)

print("Dataset shape before dropping leakage columns:", df_clean.shape)
print("Dataset shape after dropping leakage columns:", df_clean_model.shape)

print("\nColumns removed:")
print(leakage_cols)

Dataset shape before dropping leakage columns: (117396, 38)
Dataset shape after dropping leakage columns: (117396, 36)

Columns removed:
['reservation_status', 'reservation_status_date']


## 15. Final Data Quality Check

After completing the cleaning process, the final modeling dataset is checked again to confirm its quality.

This step checks:

- The final dataset shape.
- Remaining missing values.
- Exact duplicated rows kept in the dataset.
- Data types after cleaning.
- A preview of the cleaned dataset.

This final check ensures that the dataset is ready to be saved and used for preprocessing and model training.

In [20]:
# Final data quality check

print("Cleaned modeling dataset shape:", df_clean_model.shape)

print("\nMissing values after cleaning:")
print(df_clean_model.isnull().sum().sort_values(ascending=False))

print("\nExact duplicated rows kept in final dataset:", df_clean_model.duplicated().sum())

print("\nData types after cleaning:")
print(df_clean_model.dtypes)

# Preview cleaned modeling dataset
df_clean_model.head()

Cleaned modeling dataset shape: (117396, 36)

Missing values after cleaning:
hotel                             0
is_canceled                       0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces       0
total_of_special_requests         0
has_agent                         0
has_company                       0
arrival_month_num                 0
arrival_date                      0
total_guests                      0
reserved_room_type                0
previous_bookings_not_canceled    0
previous_cancellations            0
stays_in_week_nights              0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_mon

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,customer_type,adr,required_car_parking_spaces,total_of_special_requests,has_agent,has_company,arrival_month_num,arrival_date,total_guests,total_nights
0,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,0,0,7,2015-07-01,1,1
1,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,Transient,75.0,0,0,1,0,7,2015-07-01,1,1
2,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,1,0,7,2015-07-01,2,2
3,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,Transient,98.0,0,1,1,0,7,2015-07-01,2,2
4,Resort Hotel,0,0,2015,July,27,1,0,2,2,...,Transient,107.0,0,0,0,0,7,2015-07-01,2,2


## 16. Save the Cleaned Dataset

Finally, the cleaned modeling dataset is saved as a new CSV file.

This saved file will be used in the next notebook for preprocessing and model training.

The final dataset has already been cleaned, checked for missing values, and prepared to avoid data leakage.

In [21]:
# Save the cleaned modeling dataset

df_clean_model.to_csv("../Data/hotel_bookings_cleaned_model.csv", index=False)

print("Cleaned modeling dataset saved as: ../Data/hotel_bookings_cleaned_model.csv")

Cleaned modeling dataset saved as: hotel_bookings_cleaned_model.csv


## Conclusion

In this notebook, the hotel booking dataset was cleaned and prepared for the modeling stage.

The main cleaning tasks included handling missing values, converting basic data types, standardizing text columns, handling undefined categorical values, creating date-related variables, creating booking-related variables, removing logically invalid records, handling abnormal ADR values, checking lead time and guest-related values, and removing potential data leakage columns.

Exact duplicated rows were checked but not removed because the dataset does not include a unique booking ID or customer ID. Therefore, identical rows may represent different bookings with the same characteristics rather than true duplicated records.

The final modeling dataset contains no missing values and excludes columns that could cause data leakage, such as `reservation_status` and `reservation_status_date`.

The cleaned dataset was saved as `hotel_bookings_cleaned_model.csv` and will be used in the next notebook for preprocessing and model training.
